In [3]:
import requests
import pandas as pd
import geopandas as gpd
import folium
import pyproj
url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/RJ/municipios"

resposta = requests.get(url)
dados = resposta.json()

df_ibge = pd.DataFrame(dados)
df_ibge = df_ibge[["id", "nome"]]
df_ibge.columns = ["geocodigo", "municipio"]
gdf_muni = gpd.read_file("municipios.shp")

gdf_muni["geocodigo"] = gdf_muni["geocodigo"].astype(str)
df_ibge["geocodigo"] = df_ibge["geocodigo"].astype(str)
gdf_final = gdf_muni.merge(df_ibge, on="geocodigo")

gdf_final = gdf_final.set_crs(epsg=4674)
gdf_final.to_crs(epsg=4326)

centro = [-22.5, -43.3]

mapa = folium.Map(location=centro, zoom_start=7)

folium.GeoJson(gdf_final).add_to(mapa)

mapa.save(r"C:\Users\Alberto\Downloads\mapa_municipios_rj.html")


In [2]:
import folium
import pandas as pd
import geopandas as gpd
import requests
import pyproj

url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/RJ/municipios"

resposta = requests.get(url)
dados = resposta.json()

df_ibge = pd.DataFrame(dados)
df_ibge = df_ibge[["id", "nome"]]
df_ibge.columns = ["geocodigo", "municipio"]
gdf_muni = gpd.read_file("municipios.shp")

gdf_muni["geocodigo"] = gdf_muni["geocodigo"].astype(str)
df_ibge["geocodigo"] = df_ibge["geocodigo"].astype(str)

gdf_final = gdf_muni.merge(df_ibge, on="geocodigo")

gdf_final = gdf_final.set_crs(epsg=4674)
gdf_final.to_crs(epsg=4326)

centro = [-22.5, -43.3]

mapa = folium.Map(location = centro, zoom_start=7)
folium.GeoJson(gdf_final).add_to(mapa)

mapa.save(r"C:\Users\Alberto\Downloads\mapa_municipios_rj.html")


In [12]:
url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/RJ/municipios"

resposta = requests.get(url)
dados = resposta.json()

df_ibge = pd.DataFrame(dados)
df_ibge = df_ibge[["id", "nome"]]
df_ibge.columns = ["geocodigo", "municipio"]
gdf_muni = gpd.read_file("municipios.shp")

gdf_muni["geocodigo"] = gdf_muni["geocodigo"].astype(str)
df_ibge["geocodigo"] = df_ibge["geocodigo"].astype(str)

gdf_final = gdf_muni.merge(df_ibge, on="geocodigo")
gdf_final = gdf_final.set_crs(epsg=4674)
gdf_final = gdf_final.to_crs(epsg= 4326)

centro = [-22.5, -43.3]
mapa = folium.Map(location = centro, zoomstarts = 7)
folium.GeoJson(gdf_final).add_to(mapa)

mapa.save(r"C:\Users\Alberto\Downloads\mapa_municipios_rj.html")

In [3]:
import pandas as pd
import geopandas as gpd
import requests
import pyproj
import os
import tkinter as tk
from tkinter import ttk, messagebox
import ee
import geemap
import webbrowser
import ee
import folium
ee.Authenticate()
import geemap
ee.Initialize()

path_municipios_file = os.path.join(os.getcwd(), "municipios.shp")
arquivo_muni = gpd.read_file(path_municipios_file)
arquivo_muni = arquivo_muni.to_crs(epsg=4674)

def carregar_estados(evento):
    regiao = combo_regiao.get().lower()
    estados = arquivo_muni[arquivo_muni["regiao"].str.lower() == regiao]["uf"].unique()
    if len(estados) == 0:
        messagebox.showerror("Erro!", "Região Inválida!")
        return
        
    combo_estado["values"] = sorted(estados)
    combo_estado.set("")
    combo_municipio.set("")
    combo_municipio["values"] = []
    
def carregar_municipios(event):
    regiao = combo_regiao.get().lower()
    uf = combo_estado.get().upper()
    municipios = arquivo_muni[(arquivo_muni["regiao"].str.lower() == regiao) & (arquivo_muni["uf"].str.upper()== uf)]["municipio"].str.capitalize().unique()
    combo_municipio["values"] = sorted(municipios)
    combo_municipio.set("")

def processar():
    regiao = combo_regiao.get().lower()
    uf = combo_estado.get().upper()
    municipio = combo_municipio.get().lower()
    if regiao == "" or uf == "" or municipio == "":
        messagebox.showerror("Erro", "preencha todas as opções")
        return

    municipio_escolhido = arquivo_muni[(arquivo_muni["regiao"].str.lower()== regiao) &
        (arquivo_muni["uf"].str.upper()==uf) & 
        (arquivo_muni["municipio"].str.lower()==municipio)]

    if municipio_escolhido.empty:
        messagebox.showerror("Erro", "municipio não encontrado")
        return
            
    municipio_escolhido.to_file("municipio_escolhido.shp", driver="ESRI Shapefile")

    municipio_escolhido = municipio_escolhido.to_crs(epsg=4326)
    
    gdf = gpd.read_file("municipio_escolhido.shp")
    geom = geemap.geopandas_to_ee(gdf)

    dataset = ee.Image("ESA/WorldCover/v100/2020")
    recorte = dataset.clip(geom)

    Map = geemap.Map(plugin_Draw=False)

    Map.addLayer(recorte, {}, "Uso do solo")
    Map.addLayer(geom,{}, "Muncicipio")

    Map

    arquivo = "uso_solo.tif"

    if os.path.exists(arquivo):
        os.remove(arquivo)

    geemap.ee_export_image(
        recorte,
        filename="uso_solo.tif",
        scale=75,
        region=geom.geometry(),
        file_per_band=False
    )
    
    Map = geemap.Map(plugin_Draw=False)

    Map.center_object(geom, 10)

    vis = {
    "min": 10,
    "max": 100,
    "palette": [
        "006400",
        "ffbb22",
        "ffff4c",
        "f096ff",
        "fa0000",
        "b4b4b4"
    ]
    }

    Map.addLayer(recorte, vis, "Uso do solo")
    Map.addLayer(geom, {}, "Município")

    Map.add_legend(
    title="Uso do solo",
    legend_dict={
        "Floresta": "#006400",
        "Agricultura": "#ffbb22",
        "Pastagem": "#ffff4c",
        "Urbano": "#f096ff",
        "Construído": "#fa0000",
        "Solo exposto": "#b4b4b4"
    }
    )

    Map.save(r"C:\Users\Alberto\Downloads\mapa_uso_solo.html")

    messagebox.showinfo("Sucesso!")
    arquivo_html = r"C:\Users\Alberto\Downloads\mapa_uso_solo.html"
    webbrowser.open(arquivo_html)

janela = tk.Tk()
janela.title("Escolha o município que você deseja")
janela.geometry("1000x840")

style = ttk.Style()
style.configure("TLabel", font=("Segoe UI", 17))
style.configure("Combo", font=("Segoe UI", 13))
style.configure("TButton", font=("Segoe UI", 17))

ttk.Label(janela, text="Região").pack(pady=5)
combo_regiao = ttk.Combobox(janela, values=sorted(arquivo_muni["regiao"].str.capitalize().unique()),state="readonly")
combo_regiao.pack()
combo_regiao.bind("<<ComboboxSelected>>", carregar_estados)

ttk.Label(janela, text="Estado").pack(pady=5)
combo_estado = ttk.Combobox(janela, state="readonly")
combo_estado.pack()
combo_estado.bind("<<ComboboxSelected>>", carregar_municipios)

ttk.Label(janela, text="Município").pack(pady=5)
combo_municipio = ttk.Combobox(janela, state = "readonly")
combo_municipio.pack()

ttk.Button(janela, text="Processar", command = processar).pack(pady=20)
janela.mainloop()



In [4]:
import ee
ee.Authenticate()
import geemap
ee.Initialize()

In [3]:
import ee

ee.Initialize()

img = ee.Image("ESA/WorldCover/v100/2020")
print(img.getInfo())

{'type': 'Image', 'bands': [{'id': 'Map', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [4320000, 1728000], 'crs': 'EPSG:4326', 'crs_transform': [8.333333333333333e-05, 0, -180, 0, -8.333333333333333e-05, 84]}], 'version': 1685065671563344, 'id': 'ESA/WorldCover/v100/2020', 'properties': {'system:time_start': 1577836800000, 'Map_class_names': ['Tree cover', 'Shrubland', 'Grassland', 'Cropland', 'Built-up', 'Bare / sparse vegetation', 'Snow and ice', 'Permanent water bodies', 'Herbaceous wetland', 'Mangroves', 'Moss and lichen'], 'system:footprint': {'type': 'LinearRing', 'coordinates': [[-180, -90], [180, -90], [180, 90], [-180, 90], [-180, -90]]}, 'system:time_end': 1609459200000, 'Map_class_palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064c8', '0096a0', '00cf75', 'fae6a0'], 'Map_class_values': [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100], 'system:asset_size': 109355097577, 'system:index': '2020'}}


In [ ]:
path_municipios_file = os.path.join(os.getcwd(), "municipios.shp")
arquivo_muni = gpd.read_file(path_municipios_file)
arquivo_muni = arquivo_muni.to_crs(epsg=4674)

def carregar_estados(evento):
    regiao = combo_regiao.get().lower()
    estados = arquivo_muni[arquivo_muni["regiao"].str.lower() == regiao]["uf"].unique()
    if len(estados) == 0:
        messagebox.showerror("Erro!", "Região Inválida!")
        return
        
    combo_estado["values"] = sorted(estados)
    combo_estado.set("")
    combo_municipio.set("")
    combo_municipio["values"] = []
    
def carregar_municipios(event):
    regiao = combo_regiao.get().lower()
    uf = combo_estado.get().upper()
    municipios = arquivo_muni[(arquivo_muni["regiao"].str.lower() == regiao) & (arquivo_muni["uf"].str.upper()== uf)]["municipio"].str.capitalize().unique()
    combo_municipio["values"] = sorted(municipios)
    combo_municipio.set("")

def processar():
    regiao = combo_regiao.get().lower()
    uf = combo_estado.get().upper()
    municipio = combo_municipio.get().lower()
    if regiao == "" or uf == "" or municipio == "":
        messagebox.showerror("Erro", "preencha todas as opções")
        return

    municipio_escolhido = arquivo_muni[(arquivo_muni["regiao"].str.lower()== regiao) &
        (arquivo_muni["uf"].str.upper()==uf) & 
        (arquivo_muni["municipio"].str.lower()==municipio)]

    if municipio_escolhido.empty:
        messagebox.showerror("Erro", "municipio não encontrado")
        return
            
    municipio_escolhido.to_file("municipio_escolhido.shp", driver="ESRI Shapefile")

    municipio_escolhido = municipio_escolhido.to_crs(epsg=4326)
    
    gdf = gpd.read_file("municipio_escolhido.shp")
    gdf["geometry"] = gdf["geometry"].simplify(0.001)
    geom = geemap.geopandas_to_ee(gdf)

    dataset = ee.Image("ESA/WorldCover/v100/2020")
    recorte = dataset.clip(geom)

    # 🔥 Criar mapa Folium puro
    Map = folium.Map(
    location=[gdf.geometry.centroid.y.mean(),
              gdf.geometry.centroid.x.mean()],
    zoom_start=10
    )

# 🔹 Converter imagem GEE para tiles
    map_id = recorte.getMapId({
    "min": 10,
    "max": 100,
    "palette": [
        "006400",
        "ffbb22",
        "ffff4c",
        "f096ff",
        "fa0000",
        "b4b4b4"
    ]
    }
    )

# 🔹 Adicionar raster como camada
    folium.TileLayer(
    tiles=map_id["tile_fetcher"].url_format,
    attr="ESA WorldCover",
    name="Uso do solo",
    overlay=True,
    control=True
    ).add_to(Map)

# 🔹 Adicionar limite do município
    folium.GeoJson(gdf).add_to(Map)

# 🔹 Controle de camadas
    folium.LayerControl().add_to(Map)

# 🔹 Salvar
    Map.save(r"C:\Users\Alberto\Downloads\mapa_uso_solo.html")

    messagebox.showinfo("Sucesso!")
    arquivo_html = r"C:\Users\Alberto\Downloads\mapa_uso_solo.html"
    webbrowser.open(arquivo_html)

janela = tk.Tk()
janela.title("Escolha o município que você deseja")
janela.geometry("1000x840")

style = ttk.Style()
style.configure("TLabel", font=("Segoe UI", 17))
style.configure("Combo", font=("Segoe UI", 13))
style.configure("TButton", font=("Segoe UI", 17))

ttk.Label(janela, text="Região").pack(pady=5)
combo_regiao = ttk.Combobox(janela, values=sorted(arquivo_muni["regiao"].str.capitalize().unique()),state="readonly")
combo_regiao.pack()
combo_regiao.bind("<<ComboboxSelected>>", carregar_estados)

ttk.Label(janela, text="Estado").pack(pady=5)
combo_estado = ttk.Combobox(janela, state="readonly")
combo_estado.pack()
combo_estado.bind("<<ComboboxSelected>>", carregar_municipios)

ttk.Label(janela, text="Município").pack(pady=5)
combo_municipio = ttk.Combobox(janela, state = "readonly")
combo_municipio.pack()

ttk.Button(janela, text="Processar", command = processar).pack(pady=20)
janela.mainloop()
